# Olist E-commerce SQL Business Analysis

## Project Overview

This project uses SQL to analyze the Brazilian Olist E-commerce dataset.  
The goal is to answer business questions related to revenue, sellers, customers, product categories, payments, reviews, and delivery performance.

This project demonstrates:
- SQL joins
- Aggregations
- Grouping and ordering
- Date functions
- Business insight generation
- SQL analysis inside Jupyter Notebook

## 1. Import Libraries

In [ ]:
import sqlite3
import pandas as pd
import os

## 2. Load CSV Files and Create SQLite Database

In [ ]:
# Update this path if your files are in a different folder
DATA_PATH = "data/Raw/archive (11)"

# If the path above does not exist, try alternative common paths
if not os.path.exists(DATA_PATH):
    if os.path.exists("data/Raw"):
        DATA_PATH = "data/Raw"
    elif os.path.exists("data/raw"):
        DATA_PATH = "data/raw"

print("Using data path:", DATA_PATH)

files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

# Create SQLite database
conn = sqlite3.connect("olist_sql_project.db")

# Load each CSV into SQLite
for table_name, file_name in files.items():
    file_path = os.path.join(DATA_PATH, file_name)
    df = pd.read_csv(file_path)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"{table_name}: {df.shape}")

print("Database created successfully.")

## 3. Check Available Tables

In [ ]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql(query, conn)

## 4. Helper Function to Run SQL Queries

In [ ]:
def run_query(query):
    return pd.read_sql(query, conn)

## Business Question 1: Top 10 Sellers by Revenue

**Question:** Which sellers generated the highest revenue?

In [ ]:
query = """
SELECT
    oi.seller_id,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
GROUP BY oi.seller_id
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_query(query)

### Insight

The top sellers generated the highest revenue, suggesting that a small group of sellers contributes significantly to overall sales performance.

## Business Question 2: Top 10 Product Categories by Revenue

**Question:** Which product categories generated the highest revenue?

In [ ]:
query = """
SELECT
    p.product_category_name,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_query(query)

### Insight

The highest-revenue product categories show where customer demand is strongest and where Olist may prioritize marketing and inventory planning.

## Business Question 3: Top 10 States by Number of Orders

**Question:** Which customer states generated the highest number of orders?

In [ ]:
query = """
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY total_orders DESC
LIMIT 10;
"""

run_query(query)

### Insight

The states with the highest order volume represent Olist's strongest customer markets and may be important for logistics and customer acquisition.

## Business Question 4: Top 10 States by Revenue

**Question:** Which customer states generated the highest revenue?

In [ ]:
query = """
SELECT
    c.customer_state,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_state
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_query(query)

### Insight

The highest-revenue states are key markets where Olist can focus sales strategies and operational resources.

## Business Question 5: Payment Method Distribution

**Question:** Which payment methods are used most frequently?

In [ ]:
query = """
SELECT
    payment_type,
    COUNT(*) AS payment_count
FROM order_payments
GROUP BY payment_type
ORDER BY payment_count DESC;
"""

run_query(query)

### Insight

The most frequently used payment method reflects customer payment preferences and can help Olist optimize the checkout experience.

## Business Question 6: Revenue by Payment Method

**Question:** Which payment methods generated the highest payment value?

In [ ]:
query = """
SELECT
    payment_type,
    ROUND(SUM(payment_value), 2) AS total_payment_value
FROM order_payments
GROUP BY payment_type
ORDER BY total_payment_value DESC;
"""

run_query(query)

### Insight

Payment methods with the highest total value show which payment channels are most important for revenue generation.

## Business Question 7: Customer Review Score Distribution

**Question:** How are customer review scores distributed?

In [ ]:
query = """
SELECT
    review_score,
    COUNT(*) AS review_count
FROM order_reviews
GROUP BY review_score
ORDER BY review_score;
"""

run_query(query)

### Insight

The distribution of review scores helps evaluate overall customer satisfaction and identify whether most customers had positive or negative experiences.

## Business Question 8: Average Review Score by Product Category

**Question:** Which product categories have the highest average review score?

In [ ]:
query = """
SELECT
    p.product_category_name,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM order_reviews r
JOIN orders o
    ON r.order_id = o.order_id
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY avg_review_score DESC
LIMIT 10;
"""

run_query(query)

### Insight

Product categories with the highest average review scores indicate strong customer satisfaction and potentially higher product quality.

## Business Question 9: Monthly Revenue Trend

**Question:** How did revenue change over time by month?

In [ ]:
query = """
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS order_month,
    ROUND(SUM(oi.price), 2) AS monthly_revenue
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY order_month
ORDER BY order_month;
"""

run_query(query)

### Insight

The monthly revenue trend helps identify business growth patterns, seasonality, and changes in sales performance over time.

## Business Question 10: Monthly Order Volume Trend

**Question:** How did order volume change over time by month?

In [ ]:
query = """
SELECT
    strftime('%Y-%m', order_purchase_timestamp) AS order_month,
    COUNT(DISTINCT order_id) AS total_orders
FROM orders
GROUP BY order_month
ORDER BY order_month;
"""

run_query(query)

### Insight

Monthly order volume shows changes in customer demand over time and can support forecasting and planning.

## Business Question 11: Average Delivery Time by State

**Question:** Which states have the longest average delivery times?

In [ ]:
query = """
SELECT
    c.customer_state,
    ROUND(AVG(
        julianday(o.order_delivered_customer_date) 
        - julianday(o.order_purchase_timestamp)
    ), 2) AS avg_delivery_days
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_delivery_days DESC;
"""

run_query(query)

### Insight

States with longer average delivery times may require logistics improvements to increase customer satisfaction.

## Business Question 12: Top 10 Cities by Number of Orders

**Question:** Which cities generated the highest number of orders?

In [ ]:
query = """
SELECT
    c.customer_city,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_city
ORDER BY total_orders DESC
LIMIT 10;
"""

run_query(query)

### Insight

Cities with the highest order volume are important customer hubs and may be strong targets for marketing and delivery optimization.

## Business Question 13: Top 10 Cities by Revenue

**Question:** Which cities generated the highest revenue?

In [ ]:
query = """
SELECT
    c.customer_city,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_city
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_query(query)

### Insight

Cities with the highest revenue are valuable markets where Olist may focus growth strategies.

## Business Question 14: Top 10 Customers by Spending

**Question:** Which customers spent the most?

In [ ]:
query = """
SELECT
    c.customer_unique_id,
    ROUND(SUM(oi.price), 2) AS total_spent
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_unique_id
ORDER BY total_spent DESC
LIMIT 10;
"""

run_query(query)

### Insight

Top-spending customers represent high-value customers who may be important for loyalty and retention strategies.

## Business Question 15: Order Status Distribution

**Question:** What is the distribution of order statuses?

In [ ]:
query = """
SELECT
    order_status,
    COUNT(*) AS order_count
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;
"""

run_query(query)

### Insight

Order status distribution helps assess operational performance and identify potential issues such as cancellations or unavailable orders.

# Conclusion

This SQL analysis explored key business questions using the Olist E-commerce dataset.

## Key Findings

- A small number of sellers generated a large share of total revenue.
- Some product categories consistently outperformed others in revenue.
- São Paulo and other major states represented the largest markets by both orders and revenue.
- Credit card was the dominant payment method.
- Most customer review scores were positive, suggesting generally strong customer satisfaction.
- Delivery time varied by state, indicating opportunities for logistics improvement.

## Skills Demonstrated

- SQL joins
- SQL aggregation
- Grouping and sorting
- Date-based analysis
- Business insight generation
- SQL analysis using Jupyter Notebook

## Close Database Connection

In [ ]:
conn.close()